### Tests for Soroe

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyAPES.microclimate.radiation_old import solar_angles
from pyAPES.utils.constants import DEG_TO_RAD


In [2]:
from pyAPES.leaf.photo import Vcmax_from_Nleaf

x = Vcmax_from_Nleaf(2.0, 'Temperate broadleaf')

In [6]:
1.9 * x

125.704

In [ ]:
a = [5.9, 7.3, 8.75, 9.7]
b = [9.6, 3.2, -0.5, 4.5, 7.5]

print(np.mean(a), np.std(b), np.mean(b), np.std(b))


In [ ]:
raw = pd.read_csv(r'data\tmp\soroe_laidev.txt', sep=',', skipinitialspace=True)

# restrict to doy [0, 310]
raw = raw[raw['doy'] <= 325].copy()

# inverted normalization: minimum contributes to maximum LAI and maximum to minimum LAI
# use 5th percentile as the lower reference and actual max as upper reference
val_min = np.percentile(raw['value'], 5)
val_max = raw['value'].max()
raw['rLAI'] = 1 - (raw['value'] - val_min) / (val_max - val_min)
raw['rLAI'] = raw['rLAI'].clip(0, 1)

met = pd.read_csv(r'forcing/Soroe/DK-Sor_forcing_2015-2020.dat', sep=';')

# compute solar angles
lat = 55.486 # N
lon = 11.644 # E
timeoffset = 1.15 # accounts for +1 UTC and 15min correction due to timestep being ENDTIME of 30min period

jday = met['doy'] + met['hour'] / 24 + met['minute'] / (24*60)

jday = jday.values

zen, azim, decl, sunrise, sunset, daylength = solar_angles(lat, lon, jday, timezone=timeoffset)
elev_angle = 90 - zen / DEG_TO_RAD

met['daylength'] = daylength / 60 # hours
met['elev_angle'] = elev_angle


In [ ]:
plt.plot(raw['doy'], raw['rLAI'], '.-')

In [ ]:
met2018 = (met.loc[met['year'] == 2018, ['doy', 'DDsum', 'Tdaily', 'daylength']]
             .groupby('doy')
             .agg({'DDsum': 'last', 'Tdaily': 'first', 'daylength': 'max'}))

In [ ]:
met2018.head()

In [ ]:
from scipy.optimize import differential_evolution

def simulate_rLAI(params, met2018):
    """Simulate relative LAI for each day in met2018.
    
    Uses met2018['DDsum'] directly. Tbase is fixed at 5.0.
    params: [lai_min, ddo, ddmat, sdl, sdur]
    """
    lai_min, ddo, ddmat, sdl, sdur = params

    # senescence onset: last doy on the descending side where daylength >= sdl
    doys = met2018.index.values
    dl = met2018['daylength'].values
    # only look at post-solstice (doy > 172) to avoid spring false-positive
    post_sol = doys > 172
    mask = (dl >= sdl) & post_sol
    sso = int(doys[mask][-1]) if mask.any() else 200

    rLAI = []

    for doy_i, row in met2018.iterrows():
        DDsum = row['DDsum']

        # spring growth phase
        if DDsum <= ddo:
            f = lai_min
        else:
            f = min(1.0, lai_min + (1.0 - lai_min) * (DDsum - ddo) / (ddmat - ddo))

        # autumn senescence: linear decline over sdur days
        if doy_i > sso:
            f = 1.0 - (1.0 - lai_min) * min(1.0, (doy_i - sso) / sdur)

        rLAI.append(f)

    return pd.Series(rLAI, index=met2018.index)


def objective(params):
    lai_min, ddo, ddmat, sdl, sdur = params
    if ddmat <= ddo or lai_min <= 0 or lai_min >= 1 or sdur <= 0:
        return 1e9
    sim = simulate_rLAI(params, met2018)
    sim_at_obs = np.interp(raw['doy'].values, sim.index.values, sim.values)
    return np.sum((sim_at_obs - raw['rLAI'].values) ** 2)


# --- bounds ---
# [lai_min,      ddo,       ddmat,      sdl,      sdur ]
bounds = [(0.01, 0.5), (20, 200), (100, 600), (8, 14), (10, 120)]

# differential_evolution: derivative-free global search
# works correctly with the discrete step-function nature of sso(sdl)
result = differential_evolution(objective, bounds, seed=42,
                                maxiter=1000, tol=1e-6, polish=True)

lai_min_opt, ddo_opt, ddmat_opt, sdl_opt, sdur_opt = result.x

print("Optimized parameters (Tbase fixed at 5.0 °C):")
print(f"  lai_min = {lai_min_opt:.3f}")
print(f"  ddo     = {ddo_opt:.1f}  °C·day  (budburst)")
print(f"  ddmat   = {ddmat_opt:.1f}  °C·day  (full canopy)")
print(f"  sdl     = {sdl_opt:.2f} h  (senescence daylength threshold)")
print(f"  sdur    = {sdur_opt:.1f}  days  (senescence duration)")
print(f"  SSE     = {result.fun:.4f}")

# --- plot ---
sim_opt = simulate_rLAI(result.x, met2018)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(raw['doy'], raw['rLAI'], 'o', ms=4, label='observed (2018)')
ax.plot(sim_opt.index, sim_opt.values, '-', lw=2, label='fitted')
ax.set_xlabel('Day of year')
ax.set_ylabel('Relative LAI')
ax.legend()
plt.tight_layout()
plt.show()


Leaf Nitrogen and Vcmax

In [ ]:
def Vcmax_from_Nleaf(N, pft):
    #Kattge et al. 2009 GCB Table 2
    pft_params = {
        "Tropical trees (oxisols)": {"iV": 1.99, "SD_iV": 5.14, "sV": 10.71, "SD_sV": 1.90, "corr": -0.93},
          

        "Tropical trees (nonoxisols)": {"iV": 6.35, "SD_iV": 5.52, "sV": 25.88, "SD_sV": 5.16, "corr": -0.93},

        "Temperate broadleaved trees": {"iV": 5.40, "SD_iV": 1.74, "sV": 30.38, "SD_sV": 1.34, "corr": -0.90},

        "Coniferous trees": {"iV": 34.05, "SD_iV": 5.90, "sV": 9.71, "SD_sV": 2.26, "corr": -0.91},

        "Shrubs": {"iV": 4.61, "SD_iV": 8.22, "sV": 30.20, "SD_sV": 4.77, "corr": -0.93},

        "C3 herbaceous": {"iV": 23.74, "SD_iV": 6.87, "sV": 28.17, "SD_sV": 4.67, "corr": -0.96},
        "C3 crops": {"iV": 22.22, "SD_iV": 16.74, "sV": 41.27, "SD_sV": 12.53, "corr": -0.96},
    }

    a0 = pft_params[pft]['iV']
    a1 = pft_params[pft]['sV']
    
    vmax25 = a0 + a1 * N
    return vmax25

        # if 'kn' in self.photop0:
        #     kn = self.photop0['kn']
        #     Lc = np.flipud(np.cumsum(np.flipud(self.lad*self.dz)))
        #     Lc = Lc / np.maximum(Lc[0], EPS)
        #     f = np.exp(-kn*Lc)
def canopygradient(kn, Lc):
    Lc = Lc / np.maximum(Lc[-1], 0.0001)
    print(Lc)
    f = np.exp(-kn*Lc)
    return f



In [ ]:
Lc = [0.0, 2.5, 5.0]
kn = 0.5
Ntop = 2.5

N = Ntop*canopygradient(kn, Lc)

Vmax25 = Vcmax_from_Nleaf(2.5, 'Temperate broadleaved trees')
print(Vmax25)


In [ ]:
0.02*85